# Forecasting realized volatility with GNNHAR

**Replication notebook** · Chao Zhang, Xingyue Pu, Mihai Cucuringu, and Xiaowen Dong, *International Journal of Forecasting* 41 (2025), 377–397. [Published article](https://doi.org/10.1016/j.ijforecast.2024.09.002) · [Author-hosted PDF](https://web.media.mit.edu/~xdong/paper/ijf25.pdf) · [Released code at the audited commit](https://github.com/chaozhang-ox/GNNHAR/tree/98a85b0802bf9336bb68196013c7bfbfa92592f8)

**Research question.** Does adding other assets' past volatility improve forecasts beyond an asset's own daily, weekly, and monthly history? We compare two linear models and three graph depths, each under two training losses. All variants predict the same realized variance on the same held-out origins.

The saved output includes a **public-data replication of the method on 30 individual U.S. stocks** and a quote-to-variance check on official LOBSTER data. The paper's empirical benchmark comes from 27 individual Dow stocks with licensed LOBSTER inputs. Use `SOURCE = "lobster"` with those files to rerun the design on the original universe.

## 1. Run setup

From this folder, create an environment with Python 3.10 or 3.11.

```sh
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
python -m ipykernel install --user --name gnnhar --display-name "Python (GNNHAR)"
```

Open this notebook in Jupyter or VS Code, choose **Python (GNNHAR)**, then run all cells from the project folder. The default run caches three LOBSTER sample archives and two public stock-panel CSVs, verifying each SHA-256. To rerun the four-index benchmark, set `SOURCE = "oxford"`. If its archive server is unavailable, download the [pinned snapshot](https://web.archive.org/web/20220301022212id_/https://realized.oxford-man.ox.ac.uk/images/oxfordmanrealizedvolatilityindices.zip) separately and place it at `data/oxfordmanrealizedvolatilityindices.zip`. The run settings and time span are shown below the forecasts.

In [1]:
from copy import deepcopy
from dataclasses import dataclass, replace
from hashlib import sha256
from os import cpu_count
from pathlib import Path
from shutil import which
from subprocess import run
from urllib.error import URLError
from urllib.request import Request, urlopen
from warnings import catch_warnings, simplefilter
from zipfile import ZipFile

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
from IPython.display import display
from joblib import Parallel, delayed, parallel_config
from scipy.stats import t as student_t
from sklearn.covariance import GraphicalLasso, GraphicalLassoCV
from sklearn.exceptions import ConvergenceWarning
from torch import nn

torch.set_num_threads(1)

## 2. Data and why these inputs

| Research input | Paper mode | Public mode |
|---|---|---|
| Nodes | 27 DJIA constituents listed in Appendix A | DJIA, S&P 500, Nasdaq Composite, Russell 2000 |
| Forecast target | LOBSTER-derived intraday realized variance | Oxford-Man Realized Library `rv5_ss`, subsampled 5-minute realized variance |
| Graph input | Same-session daily returns from the stock data | Oxford-Man `open_to_close` for the same indices |
| Dates | July 2007 to June 2021; forecasts from July 2011 | Same requested endpoints; only jointly observed sessions |

I chose Oxford-Man because it preserves **measured intraday variance and a separate same-session return series**, both of which the experiment needs. Daily squared returns alone would replace the paper's target with a noisier single-return proxy. The four indices share U.S. sessions, include the Dow itself, and span large caps, small caps, and Nasdaq listings. Their constituents overlap, so any improvement can reflect common exposure and aggregation. The index run cannot identify spillovers between the paper's 27 individual companies. Section 2.2 loads a separate public panel of 30 individual stocks. Each horizon counts jointly observed sessions; missing or invalid rows are reported and are never forward-filled.

**Licensed inputs.** The [authors' data statement](https://github.com/chaozhang-ox/GNNHAR/tree/98a85b0802bf9336bb68196013c7bfbfa92592f8#data) identifies [LOBSTER](https://lobsterdata.com/) as a purchased source. Place the original `DJIA_var_FH1.csv` and `DJIA_ret_FH1.csv` in `data/lobster/`. Each is a wide CSV with session date in column one and all 27 ticker columns. The variance file uses percent squared, the return file percent. Optionally add `SPY_var_FH1.csv` in the same units with one `SPY` column to enable the paper's stress split.

If you only have licensed minute quotes, use `make_lobster_panels(Path("data/quotes"), Path("data/lobster"))` after the next cell. Supply one `TICKER.csv` per asset with `date,minute,bid,ask`, where `minute=0` is 09:30 New York time and `minute=390` is 16:00. Use prevailing best quotes at each grid timestamp and omit shortened sessions. The function averages squared returns across five starting offsets and uses market-session midpoint returns. This follows the paper's subsampling description. The released scripts instead group consecutive blocks and winsorize returns using full-sample cutoffs. Use the authors' processed files to retain that preprocessing. The historical licensed files are required to validate the full-universe branch.

Public data attribution: Heber, Lunde, Shephard, and Sheppard (2009), *Oxford-Man Institute's Realized Library*, [archived 28 February 2022 snapshot](https://web.archive.org/web/20220301022212id_/https://realized.oxford-man.ox.ac.uk/images/oxfordmanrealizedvolatilityindices.zip). The notebook caches the archive locally.

In [2]:
OXFORD_URL = (
    "https://web.archive.org/web/20220301022212id_/"
    "https://realized.oxford-man.ox.ac.uk/images/oxfordmanrealizedvolatilityindices.zip"
)
OXFORD_SHA256 = "f620ad530e35e4e28c667cfd53a9b68e8cac42133bdf3e734614fffc58f87be1"
OXFORD_FILE = "oxfordmanrealizedvolatilityindices.zip"
US_INDICES = (".DJI", ".IXIC", ".RUT", ".SPX")
DJIA_27 = tuple(sorted("""
    AAPL AMGN AXP BA CAT CRM CSCO CVX DIS GS HD HON IBM INTC JNJ JPM KO MCD MMM
    MRK MSFT NKE PG TRV UNH VZ WMT
""".split()))
DATA_START = "2007-07-01"
TEST_START = "2011-07-01"
DATA_END = "2021-06-30"


def verified_archive(url, destination, digest):
    """
    Cache an archive only after its SHA-256 matches the documented snapshot.
    """
    if destination.exists():
        payload = destination.read_bytes()
    else:
        curl = which("curl")
        if curl:
            payload = run(
                [curl, "--fail", "--silent", "--show-error", "--location", "--max-time", "90", url],
                capture_output=True, check=True,
            ).stdout
        else:
            request = Request(url, headers={"User-Agent": "GNNHAR research notebook"})
            try:
                with urlopen(request, timeout=90) as response:
                    payload = response.read()
            except URLError as error:
                raise RuntimeError(f"Download {url} manually and save it as {destination}.") from error
    if sha256(payload).hexdigest() != digest:
        raise ValueError(f"Unexpected archive hash. Replace {destination} with the documented snapshot.")
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(payload)
    return destination


def aligned_panel(variance, returns, symbols):
    """
    Use shared observed sessions and put variance and log returns in percent units.
    """
    variance = variance.loc[DATA_START:DATA_END, list(symbols)]
    returns = returns.reindex(index=variance.index, columns=symbols)
    valid = (
        np.isfinite(variance).all(axis=1)
        & variance.gt(0).all(axis=1)
        & np.isfinite(returns).all(axis=1)
    )
    dropped = int((~valid).sum())
    variance, returns = variance.loc[valid], returns.loc[valid]
    if len(variance) < 1001 or variance.index[-1] < pd.Timestamp(DATA_END):
        raise ValueError("Need a complete final session and enough history for rolling estimation.")
    return variance, returns, dropped


def load_oxford(data_dir):
    archive = verified_archive(OXFORD_URL, data_dir / OXFORD_FILE, OXFORD_SHA256)
    columns = ["Unnamed: 0", "Symbol", "rv5_ss", "open_to_close"]
    data = pd.read_csv(archive, usecols=columns).rename(columns={"Unnamed: 0": "date"})
    data = data.loc[data["Symbol"].isin(US_INDICES)].copy()
    data["date"] = pd.to_datetime(data["date"].str[:10])
    variance = data.pivot(index="date", columns="Symbol", values="rv5_ss") * 10_000
    returns = data.pivot(index="date", columns="Symbol", values="open_to_close") * 100
    return aligned_panel(variance, returns, US_INDICES)


def read_wide(path):
    panel = pd.read_csv(path, index_col=0, parse_dates=True)
    if not isinstance(panel.index, pd.DatetimeIndex) or panel.index.has_duplicates:
        raise ValueError(f"{path} needs unique session dates in its first column.")
    return panel.astype(float).sort_index()


def load_lobster(data_dir):
    """
    Read the authors' percent-squared RV and percent-return panel format.
    """
    variance = read_wide(data_dir / "DJIA_var_FH1.csv")
    returns = read_wide(data_dir / "DJIA_ret_FH1.csv")
    if not set(DJIA_27).issubset(variance.columns) or not set(DJIA_27).issubset(returns.columns):
        raise ValueError("Both LOBSTER panels must contain all 27 paper tickers.")
    return aligned_panel(variance, returns, DJIA_27)


def quote_session(quote, allow_open_gap=False):
    """
    Compute subsampled daily RV for a complete regular session of minute quotes.
    """
    quote = quote.sort_values("minute")
    grid = quote["minute"].to_numpy()
    full_session = np.array_equal(grid, np.arange(391))
    sample_session = allow_open_gap and np.array_equal(grid, np.arange(1, 391))
    if not (full_session or sample_session):
        raise ValueError("Expected one minute quote for each minute from 09:30 to 16:00.")
    bid, ask = quote["bid"].to_numpy(), quote["ask"].to_numpy()
    if not (np.isfinite(bid).all() and np.isfinite(ask).all() and (bid > 0).all() and (ask >= bid).all()):
        raise ValueError("Expected positive, finite quotes with ask at least bid.")
    price = np.log((ask + bid) / 2) * 100
    phase_variances = [np.square(np.diff(price[offset::5])).sum() for offset in range(5)]
    return pd.Series({"variance": np.mean(phase_variances), "return": price[-1] - price[0]})


def make_lobster_panels(quotes_dir, output_dir):
    """
    Convert local minute quote files into aligned wide LOBSTER input files.
    """
    assets = {}
    for ticker in DJIA_27:
        quote = pd.read_csv(quotes_dir / f"{ticker}.csv", parse_dates=["date"])
        full_session = quote.groupby("date").filter(lambda day: len(day) == 391)
        assets[ticker] = full_session.groupby("date")[["minute", "bid", "ask"]].apply(quote_session)
    output_dir.mkdir(parents=True, exist_ok=True)
    for field, filename in (("variance", "DJIA_var_FH1.csv"), ("return", "DJIA_ret_FH1.csv")):
        panel = pd.DataFrame({ticker: daily[field] for ticker, daily in assets.items()})
        panel.sort_index().to_csv(output_dir / filename)

### 2.1 Verify variance construction on LOBSTER

The provider publishes level-one message and order-book files for **AAPL, INTC, and MSFT on 21 June 2012**. All three names are in the paper. For each minute we take the last book event at or before its clock timestamp, then compute midpoint log returns and average squared five-minute returns across five grids. The first events occur just after 09:30, so the sample calculation uses 09:31 through 16:00. This keeps the timestamps free of forward lookups. The official data validate ingestion and measurement for a single session. The Oxford-Man panel supplies the multi-year history for the public backtest. [Official sample archives](https://php.lobsterdata.com/info/DataSamples.php) · [File schema](https://php.lobsterdata.com/info/DataStructure.php)

In [3]:
SAMPLE_DATE = "2012-06-21"
SAMPLE_SHA256 = {
    "AAPL": "b50994786f2996ae3c349d6b11a07b412c981a417a17ece44f068f9e12e45058",
    "INTC": "e081fde3506cb383f38d018d4a9b49db8b0ebc059ec504c96d02bc82c32d0acd",
    "MSFT": "613f9638c95b4b04894d046f784ac7bb7cbc0744f5bd873ee2d67d2cb7570928",
}
LOBSTER_SAMPLES = "https://php.lobsterdata.com/info/sample"


def minute_quotes_from_lobster(archive):
    """
    Match each clock minute to the last observed level-one order book.
    """
    with ZipFile(archive) as bundle:
        message, = [name for name in bundle.namelist() if "_message_" in name]
        orderbook, = [name for name in bundle.namelist() if "_orderbook_" in name]
        with bundle.open(message) as handle:
            events = pd.read_csv(handle, header=None, usecols=[0])[0].to_numpy()
        with bundle.open(orderbook) as handle:
            quotes = pd.read_csv(handle, header=None, usecols=[0, 1, 2, 3])
    if len(events) != len(quotes) or not np.isfinite(events).all() or (np.diff(events) < 0).any():
        raise ValueError("Expected ordered timestamps and a matching book row for every event.")

    minutes = np.arange(391)
    clocks = 9.5 * 3600 + minutes * 60
    positions = np.searchsorted(events, clocks, side="right") - 1
    available = positions >= 0
    snapshots = quotes.iloc[positions[available]]
    if (snapshots[[1, 3]] <= 0).any().any():
        raise ValueError("Encountered a placeholder or empty best quote at a grid timestamp.")
    return pd.DataFrame({
        "minute": minutes[available],
        "bid": snapshots[2].to_numpy() / 10_000,
        "ask": snapshots[0].to_numpy() / 10_000,
    })


sample_dir = Path("data/lobster_samples")
sample_outputs, sample_paths = [], []
for ticker, digest in SAMPLE_SHA256.items():
    filename = f"LOBSTER_SampleFile_{ticker}_{SAMPLE_DATE}_1.zip"
    archive = verified_archive(f"{LOBSTER_SAMPLES}/{filename}", sample_dir / filename, digest)
    quotes = minute_quotes_from_lobster(archive)
    stats = quote_session(quotes, allow_open_gap=True)
    midpoint = (quotes["bid"] + quotes["ask"]) / 2
    sample_outputs.append({
        "ticker": ticker, "minute_quotes": len(quotes),
        "return (%)": stats["return"], "realized variance (%²)": stats["variance"],
    })
    sample_paths.append(pd.DataFrame({
        "New York session time": pd.Timestamp(SAMPLE_DATE + " 09:30") + pd.to_timedelta(quotes.minute, unit="min"),
        "Midpoint log return (%)": np.log(midpoint / midpoint.iloc[0]) * 100,
        "Ticker": ticker,
    }))

display(pd.DataFrame(sample_outputs).round(4))
sample_plot = px.line(
    pd.concat(sample_paths), x="New York session time", y="Midpoint log return (%)", color="Ticker",
    title="LOBSTER · one-minute midpoint moves on 21 June 2012",
    color_discrete_sequence=["#087f8c", "#e48237", "#7871aa"],
)
sample_plot.update_layout(template="plotly_white", hovermode="x unified")
sample_plot.show(renderer="plotly_mimetype")

,ticker,minute_quotes,return (%),realized variance (%²)
0,AAPL,390,-1.3593,0.6720
1,INTC,390,-3.1322,1.5934
2,MSFT,390,-2.9267,1.7678


### 2.2 Load a public panel of individual U.S. stocks

Clements and Perera's [replication data](https://github.com/AdamEClements/Wavelet-HAR-Reconciliation-data/tree/6e7a69a8548f9a24595f7007fb0fbbf0424b8dc3) include daily realized variance calculated from five-minute Refinitiv Datascope prices and daily returns for 30 U.S. stocks. Both CSVs use numbered series, so we match the variance and return columns by number and keep those identifiers. We do not assume they are the original Dow tickers. The files span January 2000 to May 2024 and the loader retains only the dates in the GNNHAR study window. [Data methodology](https://onlinelibrary.wiley.com/doi/10.1002/for.70113)

The cell below loads and verifies both panels. The full backtest below uses this panel by default. Set `SOURCE = "oxford"` to rerun the four-index benchmark.

In [4]:
CLEMENTS_COMMIT = "6e7a69a8548f9a24595f7007fb0fbbf0424b8dc3"
CLEMENTS_URL = (
    "https://raw.githubusercontent.com/AdamEClements/Wavelet-HAR-Reconciliation-data/"
    + CLEMENTS_COMMIT
)
CLEMENTS_FILES = {
    "rv_stocks.csv": "52a5c4b1f1d96a5244882c2b0950ea4e190b5d3e1fba9e46c43f203000bcf8d7",
    "rets_stocks.csv": "3838c3b99c2456a4e672a0d1fd27ac8ed7950b9d94df0e9eb838a4d4d5233969",
}


def read_numbered_panel(path, prefix, symbols):
    """
    Match each numbered column to a stable stock identifier and sort the dates.
    """
    panel = pd.read_csv(path, index_col="dts")
    panel.index = pd.to_datetime(panel.index, format="%d/%m/%Y")
    if panel.index.has_duplicates:
        raise ValueError(f"Duplicate session dates in {path.name}.")
    expected = [f"{prefix}{number}" for number in range(1, len(symbols) + 1)]
    if set(panel.columns) != set(expected):
        raise ValueError(f"Unexpected stock columns in {path.name}.")
    panel = panel.loc[:, expected].astype(float).sort_index()
    panel.columns = symbols
    return panel


def load_clements(data_dir):
    """
    Load verified five-minute realized variance and daily returns in percent units.
    """
    for name, digest in CLEMENTS_FILES.items():
        verified_archive(f"{CLEMENTS_URL}/{name}", data_dir / name, digest)
    symbols = [f"stock_{number:02d}" for number in range(1, 31)]
    variance = read_numbered_panel(data_dir / "rv_stocks.csv", "rv_", symbols)
    returns = read_numbered_panel(data_dir / "rets_stocks.csv", "rets_d_", symbols)
    if not variance.index.equals(returns.index):
        raise ValueError("The stock variance and returns calendars do not match.")
    return aligned_panel(variance * 10_000, returns * 100, symbols)


stock_variance, stock_returns, stock_dropped = load_clements(Path("data/clements"))
print(f"clements–perera: {len(stock_variance):,} aligned sessions · {stock_variance.shape[1]} stock series")
print(f"{stock_variance.index[0]:%Y-%m-%d} to {stock_variance.index[-1]:%Y-%m-%d}")
print(f"Rows dropped for missing or invalid values: {stock_dropped}")
display(stock_variance.iloc[:3, :4].round(4))

clements–perera: 3,482 aligned sessions · 30 stock series
2007-07-02 to 2021-06-30
Rows dropped for missing or invalid values: 0


,stock_01,stock_02,stock_03,stock_04
dts,,,,
2007-07-02,0.4775,0.6155,0.5030,0.8350
2007-07-05,1.3143,0.4402,0.4059,1.6328
2007-07-06,0.8017,0.6116,0.5841,0.4767


In [5]:
SOURCE = "clements"
DATA_DIRS = {
    "oxford": Path("data"),
    "lobster": Path("data/lobster"),
    "clements": Path("data/clements"),
}
if SOURCE not in DATA_DIRS:
    raise ValueError("Choose 'oxford', 'lobster', or 'clements'.")
DATA_DIR = DATA_DIRS[SOURCE]
TRAIN_FULL_ENSEMBLE = SOURCE == "lobster"
EXPORT_RESULTS = False
WORKERS = min(4, cpu_count() or 1)

if SOURCE == "oxford":
    variance, returns, dropped = load_oxford(DATA_DIR)
elif SOURCE == "lobster":
    variance, returns, dropped = load_lobster(DATA_DIR)
else:
    variance, returns, dropped = stock_variance, stock_returns, stock_dropped

print(f"{SOURCE}: {len(variance):,} aligned sessions · {variance.shape[1]} nodes")
print(f"{variance.index[0]:%Y-%m-%d} to {variance.index[-1]:%Y-%m-%d}")
print(f"Session rows dropped for missing or invalid observations: {dropped}")
display(variance.describe().T[["count", "mean", "std", "min", "max"]].round(3))

clements: 3,482 aligned sessions · 30 nodes
2007-07-02 to 2021-06-30
Session rows dropped for missing or invalid observations: 0


,count,mean,std,min,max
stock_01,3482.0,1.778,3.461,0.128,83.772
stock_02,3482.0,4.066,10.231,0.137,202.472
stock_03,3482.0,1.395,3.016,0.081,71.417
stock_04,3482.0,2.056,5.182,0.093,135.752
stock_05,3482.0,3.076,6.053,0.128,111.819
stock_06,3482.0,1.223,3.484,0.087,80.265
stock_07,3482.0,1.095,2.575,0.074,56.976
stock_08,3482.0,2.383,5.261,0.121,100.281
stock_09,3482.0,1.933,3.944,0.129,100.441
stock_10,3482.0,5.818,15.617,0.115,246.550


## 3. Forecast equations and graph

Use the paper's disjoint own-history features: yesterday, the mean of lags 2–5, and the mean of lags 6–22. The targets are $RV_t$ and the forward sums $sum_{k=0}^{h}RV_{t+k}$ for the paper's $h=4,21$. That gives **1, 5, and 22 sessions**, including the origin. A forecast for session $t$ uses completed sessions through $t-1$.

- **HAR** uses asset intercepts and shared coefficients on the three own-history features.
- **GHAR** adds three shared coefficients on $WX$, the neighbors' degree-normalized histories.
- **GNNHAR** replaces that linear graph term with $H^ell=operatorname{ReLU}(WH^{ell-1}Theta^ell)$ and a learned readout. The separate own-history branch remains linear before the final positivity clamp.

Graphical LASSO estimates the precision matrix of recent **daily returns**. Its nonzero off-diagonal entries define undirected edges. The audit records CV warnings and any increase in the chosen penalty needed for a converged final fit. Apply $W=D^{-1/2}AD^{-1/2}$; an isolated node receives a zero graph contribution. Edges describe conditional statistical association and can transmit messages in both directions.

The training criteria are squared error and QLIKE. The training expression $y/hat y+loghat y$ drops the terms constant in the forecast; the evaluation restores $y/hat y-log(y/hat y)-1$. This is the same minimizer for strictly positive forecasts. QLIKE weights relative errors, whereas MSE measures absolute variance errors.

In [6]:
@dataclass(frozen=True)
class ModelSpec:
    name: str
    depth: int
    graph: bool
    loss: str


MODEL_SPECS = [
    ModelSpec(name + suffix, depth, depth > 0 or name == "GHAR", loss)
    for suffix, loss in (("M", "MSE"), ("Q", "QLIKE"))
    for name, depth in (("HAR", 0), ("GHAR", 0), ("GNNHAR1L", 1), ("GNNHAR2L", 2), ("GNNHAR3L", 3))
]


def har_features(variance):
    """
    Construct disjoint lag averages available before each forecast origin.
    """
    return np.stack([
        variance.shift(1).to_numpy(),
        variance.shift(2).rolling(4).mean().to_numpy(),
        variance.shift(6).rolling(17).mean().to_numpy(),
    ], axis=-1)


def forward_variance(variance, horizon):
    return variance.rolling(horizon).sum().shift(1 - horizon).to_numpy()


def fit_graph(returns):
    """
    Estimate a binary graph and normalize by the degrees of both endpoints.
    """
    common_scale = np.sqrt(np.var(returns, axis=0).mean())
    if common_scale <= 0:
        raise ValueError("The graph window has no return variation.")
    scaled_returns = returns / common_scale
    covariance = np.cov(scaled_returns, rowvar=False, ddof=0)
    np.fill_diagonal(covariance, 0)
    diagonal_alpha = np.abs(covariance).max() * (1 + 1e-6)
    with catch_warnings(record=True) as cv_warnings:
        simplefilter("always")
        search = GraphicalLassoCV(cv=5, n_jobs=1, max_iter=300).fit(scaled_returns)

    options = [search.alpha_ * multiple for multiple in (1.0, 1.25, 1.5, 2.0, 4.0, 8.0)]
    options = sorted(set(min(penalty, diagonal_alpha) for penalty in options + [diagonal_alpha]))
    for alpha in options:
        with catch_warnings(record=True) as fit_warnings:
            simplefilter("always", ConvergenceWarning)
            estimator = GraphicalLasso(alpha=alpha, max_iter=1000, enet_tol=1e-8).fit(scaled_returns)
        if not any(issubclass(item.category, ConvergenceWarning) for item in fit_warnings):
            break
    else:
        raise RuntimeError("Could not fit a converged Graphical LASSO graph.")

    adjacency = estimator.precision_ != 0
    np.fill_diagonal(adjacency, False)
    degree = adjacency.sum(axis=1)
    scale = np.zeros(len(degree))
    np.divide(1, np.sqrt(degree), out=scale, where=degree > 0)
    graph = scale[:, None] * adjacency * scale[None, :]
    return adjacency, graph, alpha * common_scale ** 2, alpha / search.alpha_, len(cv_warnings)


def fit_ols(x, y, graph, x_test, use_graph):
    """
    Fit shared HAR slopes and asset fixed effects by within-asset demeaning.
    """
    if use_graph:
        x = np.concatenate([x, graph @ x], axis=-1)
        x_test = np.concatenate([x_test, graph @ x_test], axis=-1)
    x_mean, y_mean = x.mean(axis=0), y.mean(axis=0)
    slope = np.linalg.lstsq(
        (x - x_mean).reshape(-1, x.shape[-1]), (y - y_mean).ravel(), rcond=None
    )[0]
    prediction = (x_test - x_mean) @ slope + y_mean
    return np.where(prediction > 0, prediction, y.min(axis=0))


class GNNHAR(nn.Module):
    """
    Combine an asset's own HAR features with an optional graph branch.
    """
    def __init__(self, n_assets, spec, width):
        super().__init__()
        self.use_graph = spec.graph
        self.own = nn.Linear(3, 1, bias=False)
        self.intercept = nn.Parameter(torch.full((n_assets,), 0.15))
        channels = [3] + [width] * spec.depth
        self.layers = nn.ModuleList(
            nn.Linear(in_dim, out_dim, bias=False)
            for in_dim, out_dim in zip(channels, channels[1:])
        )
        self.readout = nn.Linear(channels[-1], 1, bias=False) if spec.graph else None
        with torch.no_grad():
            self.own.weight.copy_(torch.tensor([[0.50, 0.20, 0.15]]))
            self.own.weight.add_(0.02 * torch.randn_like(self.own.weight))
            if self.readout is not None:
                self.readout.weight.mul_(0.1)

    def forward(self, features, graph):
        raw = self.own(features).squeeze(-1) + self.intercept
        hidden = features
        if self.use_graph:
            if not self.layers:
                hidden = graph @ hidden
            for layer in self.layers:
                hidden = torch.relu(layer(graph @ hidden))
            raw = raw + self.readout(hidden).squeeze(-1)
        return raw.clamp_min(1e-4), hidden


def training_loss(actual, forecast, criterion):
    if criterion == "MSE":
        return (actual - forecast).square().mean()
    return (actual / forecast + torch.log(forecast)).mean()


def fit_neural(x, y, graph, fit, validation, spec, seed, config):
    """
    Fit one seeded model and restore the best full validation checkpoint.
    """
    torch.manual_seed(seed)
    model = GNNHAR(y.shape[1], spec, config.width)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    best_loss, best_state, stale_epochs = np.inf, None, 0
    model.train()
    for epoch in range(config.epochs):
        ordering = fit[torch.randperm(len(fit)).numpy()]
        for rows in np.array_split(ordering, np.arange(32, len(ordering), 32)):
            optimizer.zero_grad(set_to_none=True)
            loss = training_loss(y[rows], model(x[rows], graph)[0], spec.loss)
            loss.backward()
            optimizer.step()
        with torch.no_grad():
            score = training_loss(y[validation], model(x[validation], graph)[0], spec.loss).item()
        if not np.isfinite(score):
            raise FloatingPointError(f"Nonfinite validation loss for {spec.name}, seed {seed}.")
        if score < best_loss:
            best_loss, best_state, stale_epochs = score, deepcopy(model.state_dict()), 0
        else:
            stale_epochs += 1
        if stale_epochs >= config.patience:
            break
    model.load_state_dict(best_state)
    return model.eval(), epoch + 1


def mean_average_distance(hidden, adjacency):
    """
    Measure average positive cosine distance along edges with nonzero embeddings.
    """
    norm = np.linalg.norm(hidden, axis=-1)
    normal = hidden / np.where(norm > 0, norm, 1)[..., None]
    distance = np.maximum(1 - normal @ normal.swapaxes(-1, -2), 0)
    usable = adjacency & (norm[..., :, None] > 0) & (norm[..., None, :] > 0)
    positive = usable & (distance > 1e-6)
    count = positive.sum(axis=-1)
    node = np.sum(distance * positive, axis=-1) / np.maximum(count, 1)
    valid_nodes = (count > 0).sum(axis=-1)
    return np.divide(
        node.sum(axis=-1), valid_nodes, out=np.full(len(node), np.nan), where=valid_nodes > 0
    )

## 4. Walk-forward calibration

Refit monthly using up to 1,000 preceding joint sessions. Reserve 250 for neural validation and use all completed calibration labels for both linear MSE and linear QLIKE fits. Estimate the common monthly graph from past returns through the last session preceding the forecast month, as in the released code. Set `graph_train_only=True` in the configuration for a sensitivity run where graph estimation also predates neural validation.

Train with Adam ($10^{-3}$), batches of 32 dates, width 9, and validation patience 15. The saved public run uses two seeds and a short epoch cap to cover the full date range. Enable `TRAIN_FULL_ENSEMBLE` for ten seeds and a 5,000-epoch ceiling with early stopping. Linear QLIKE checkpoints are selected by calibration loss because the model uses both calibration blocks. Calibration scales come from training targets only. Each direct multi-session model fits average variance and converts its output back to a sum.

| Ambiguity resolved | Choice in this notebook |
|---|---|
| Equations vs. released cumulative lags and shared intercept | Follow the article's disjoint lags and asset-specific intercepts. The paper's lag bases are related by a linear change of coordinates, but training paths may differ. |
| Monthly refit vs. released 22-row stride | Use calendar months, matching §4.1. Initial lag warm-up can shorten the first fit. |
| Supplemental protocol's 4/21 vs. article's inclusive sums | Use 5/22 observed sessions, matching the displayed equation and conventional week/month labels. |
| Multi-day targets near split boundaries | Omit any training or validation label whose final observation reaches into the next block. Hold out test origins with incomplete end-of-sample targets. |
| QLIKE positivity and stochastic fitting | Use median scaling, positive HAR-centered initialization, a small positive floor, and fixed seeds. These choices can differ from the released script. |
| Benchmark loss aggregation | Pool unrounded stock/date losses before dividing by HAR-MSE. The authors' summary script also reports mean ticker-normalized loss and applies rounding; weighting can change the ranking. |

Each monthly model is held fixed while lag inputs update with newly observed sessions. Forward targets can cross from one test month into the next because the forecast is still indexed by the day on which it was made. The boundary audit checks the last date used for graph estimation and for each calibration label.

In [7]:
@dataclass(frozen=True)
class BacktestConfig:
    horizons: tuple[int, ...] = (1, 5, 22)
    lookback: int = 1000
    validation_days: int = 250
    epochs: int = 16
    patience: int = 15
    seeds: int = 2
    width: int = 9
    graph_train_only: bool = False
    months: tuple[str, ...] | None = None


def monthly_splits(dates, config):
    """
    Recalibrate before the first observed session of each calendar month.
    """
    test_rows = np.flatnonzero(dates >= TEST_START)
    months = dates[test_rows].to_period("M")
    for month in months.unique():
        if config.months and str(month) not in config.months:
            continue
        test = test_rows[months == month]
        start = max(22, test[0] - config.lookback)
        val_start = test[0] - config.validation_days
        if val_start <= start:
            raise ValueError(f"Insufficient calibration data for {month}.")
        yield str(month), start, val_start, test


def fit_ensemble(x, y, graph, fit, validation, test, spec, config):
    forecasts, own_path, embeddings, epochs = [], [], [], []
    for seed in range(config.seeds):
        model, epoch_count = fit_neural(x, y, graph, fit, validation, spec, seed + 1, config)
        with torch.no_grad():
            forecast, hidden = model(x[test], graph)
            zero_graph_forecast, _ = model(x[test], torch.zeros_like(graph))
        forecasts.append(forecast.numpy())
        own_path.append(zero_graph_forecast.numpy())
        if spec.depth:
            embeddings.append(hidden.numpy())
        epochs.append(epoch_count)
    hidden = np.mean(embeddings, axis=0) if embeddings else None
    return np.mean(forecasts, axis=0), np.mean(own_path, axis=0), hidden, np.mean(epochs)


def forecast_frame(dates, symbols, actual, forecast, own_path, mad, spec, horizon, month):
    return pd.DataFrame({
        "date": np.repeat(dates, len(symbols)),
        "asset": np.tile(symbols, len(dates)),
        "horizon": horizon,
        "month": month,
        "model": spec.name,
        "actual": actual.ravel(),
        "forecast": forecast.ravel(),
        "peers_off": own_path.ravel(),
        "mad": np.repeat(mad, len(symbols)),
    })


def roll_backtest(variance, returns, config):
    """
    Purge forward labels that cross a training, validation, or sample boundary.
    """
    if min(config.horizons) < 1 or config.seeds < 1 or config.epochs < 1:
        raise ValueError("Horizons, epochs, and ensemble size must be positive.")
    features = har_features(variance)
    returns = returns.to_numpy()
    dates = variance.index
    targets = {h: forward_variance(variance, h) for h in config.horizons}
    forecasts, audit = [], []
    for month, start, val_start, test in monthly_splits(dates, config):
        origin = test[0]
        graph_end = val_start if config.graph_train_only else origin
        graph_returns = returns[max(0, origin - config.lookback):graph_end]
        adjacency, graph, alpha, alpha_multiplier, cv_warnings = fit_graph(graph_returns)
        graph_tensor = torch.tensor(graph, dtype=torch.float32)
        for horizon, target in targets.items():
            usable_test = test[test + horizon <= len(dates)]
            if not len(usable_test):
                continue
            train = np.arange(start, val_start - horizon + 1)
            validation = np.arange(val_start, origin - horizon + 1)
            calibration = np.arange(start, origin - horizon + 1)
            if not len(train) or not len(validation):
                raise ValueError(f"Insufficient label history for horizon {horizon} in {month}.")
            assert train[-1] + horizon <= val_start and calibration[-1] + horizon <= origin
            scale = np.median(target[train]) / horizon
            x = torch.tensor(features / scale, dtype=torch.float32)
            y = torch.tensor(target / (horizon * scale), dtype=torch.float32)
            for spec in MODEL_SPECS:
                mad = np.full(len(usable_test), np.nan)
                if spec.depth == 0 and spec.loss == "MSE":
                    forecast = fit_ols(
                        features[calibration], target[calibration], graph,
                        features[usable_test], spec.graph,
                    )
                    own_path = np.full_like(forecast, np.nan)
                    epochs = 0.0
                else:
                    fit_rows, monitor = (train, validation) if spec.depth else (calibration, calibration)
                    forecast, own_path, hidden, epochs = fit_ensemble(
                        x, y, graph_tensor, fit_rows, monitor, usable_test, spec, config
                    )
                    forecast = forecast * (horizon * scale)
                    own_path = own_path * (horizon * scale)
                    if hidden is not None:
                        mad = mean_average_distance(hidden, adjacency)
                forecasts.append(forecast_frame(
                    dates[usable_test], variance.columns, target[usable_test],
                    forecast, own_path, mad, spec, horizon, month
                ))
                audit.append({
                    "month": month, "horizon": horizon, "model": spec.name,
                    "first_forecast": dates[origin],
                    "latest_train_target": dates[(train if spec.depth else calibration)[-1] + horizon - 1],
                    "latest_validation_target": dates[calibration[-1] + horizon - 1],
                    "latest_graph_return": dates[graph_end - 1],
                    "edges": int(adjacency.sum() // 2), "graph_alpha": alpha,
                    "graph_alpha_multiplier": alpha_multiplier, "graph_cv_warnings": cv_warnings,
                    "epochs_run_mean": epochs, "test_origins": len(usable_test),
                })
    if not forecasts:
        raise ValueError("No forecast windows matched the configuration.")
    forecast = pd.concat(forecasts, ignore_index=True)
    if not (np.isfinite(forecast["forecast"]).all() and forecast["forecast"].gt(0).all()):
        raise ValueError("Forecasts must be finite and strictly positive.")
    assert not forecast.duplicated(["date", "asset", "horizon", "model"]).any()
    return forecast, pd.DataFrame(audit)


def run_month_chunk(variance, returns, config):
    torch.set_num_threads(1)
    return roll_backtest(variance, returns, config)


def run_experiment(variance, returns, config, workers):
    """
    Distribute independent months across notebook workers.
    """
    if workers <= 1:
        return run_month_chunk(variance, returns, config)
    months = [month for month, *_ in monthly_splits(variance.index, config)]
    chunks = [tuple(months[start:start + 4]) for start in range(0, len(months), 4)]
    with parallel_config(backend="loky", inner_max_num_threads=1):
        results = Parallel(n_jobs=workers, verbose=5)(
            delayed(run_month_chunk)(variance, returns, replace(config, months=chunk))
            for chunk in chunks
        )
    forecasts, audits = zip(*results)
    return pd.concat(forecasts, ignore_index=True), pd.concat(audits, ignore_index=True)

In [8]:
config = BacktestConfig(
    seeds=10 if TRAIN_FULL_ENSEMBLE else 2,
    epochs=5000 if TRAIN_FULL_ENSEMBLE else 16,
)
forecast, audit = run_experiment(variance, returns, config, WORKERS)
for column in ("latest_train_target", "latest_validation_target", "latest_graph_return"):
    assert (audit[column] < audit["first_forecast"]).all()

coverage = audit.loc[audit.model.eq("HARM")].groupby("horizon").agg(
    monthly_fits=("month", "nunique"), evaluated_origins=("test_origins", "sum")
)
print(f"{config.seeds} seeds · {config.epochs:,} maximum epochs · CPU")
print(f"Forecast origins: {forecast.date.min():%Y-%m-%d} to {forecast.date.max():%Y-%m-%d}")
display(coverage)
display(audit.loc[audit.model.eq("GNNHAR1LQ")].head(6))
graphs = audit.drop_duplicates("month")
print(f"Graph edges per month: {graphs.edges.min()}–{graphs.edges.max()}")
print(f"Months with a stricter graph penalty: {(graphs.graph_alpha_multiplier > 1.001).sum()}")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:  1.8min


2 seeds · 16 maximum epochs · CPU
Forecast origins: 2011-07-01 to 2021-06-30


[Parallel(n_jobs=4)]: Done  30 out of  30 | elapsed:  4.8min finished


,monthly_fits,evaluated_origins
horizon,,
1,120,2484
5,120,2480
22,120,2463


,month,horizon,model,first_forecast,latest_train_target,latest_validation_target,latest_graph_return,edges,graph_alpha,graph_alpha_multiplier,graph_cv_warnings,epochs_run_mean,test_origins
7,2011-07,1,GNNHAR1LQ,2011-07-01,2010-07-02,2011-06-30,2011-06-30,47,2.441435,1.0,1,16.0,20
17,2011-07,5,GNNHAR1LQ,2011-07-01,2010-07-02,2011-06-30,2011-06-30,47,2.441435,1.0,1,16.0,20
27,2011-07,22,GNNHAR1LQ,2011-07-01,2010-07-02,2011-06-30,2011-06-30,47,2.441435,1.0,1,16.0,20
37,2011-08,1,GNNHAR1LQ,2011-08-01,2010-08-02,2011-07-29,2011-07-29,47,2.435696,1.0,1,16.0,23
47,2011-08,5,GNNHAR1LQ,2011-08-01,2010-08-02,2011-07-29,2011-07-29,47,2.435696,1.0,1,16.0,23
57,2011-08,22,GNNHAR1LQ,2011-08-01,2010-08-02,2011-07-29,2011-07-29,47,2.435696,1.0,1,16.0,23


Graph edges per month: 35–271
Months with a stricter graph penalty: 0


## 5. Score forecasts and check the paper's empirical claims

Each ratio uses the same horizon and loss for the HAR-MSE denominator. A ratio below 1 improves on that baseline. Scores for the public experiment and the paper are shown separately so that differences in the universe and optimizer budget remain visible. Published figures below are the numeric values transcribed from Table 1.

Diebold–Mariano compares the date-by-date loss differences for two depths, with a rectangular HAC window of $h-1$ lags for $h$-session targets and the Harvey small-sample correction. Positive statistics favor the shallower model. The model confidence set uses a stationary bootstrap on chronological panel loss averages, preserving shared shocks across models and accounting for overlapping horizons with mean block length $max(2,h)$. That is a documented inference choice. The authors' published script resamples ticker-level loss summaries.

In [9]:
def add_losses(forecast):
    ratio = forecast["actual"] / forecast["forecast"]
    return forecast.assign(
        MSE=(forecast["actual"] - forecast["forecast"]) ** 2,
        QLIKE=ratio - np.log(ratio) - 1,
    )


def summarize_losses(losses):
    mean = losses.groupby(["horizon", "model"])[["MSE", "QLIKE"]].mean()
    baseline = mean.xs("HARM", level="model")
    return mean.div(baseline, level="horizon").unstack("horizon").swaplevel(axis=1).sort_index(axis=1)


def dm_test(first, second, horizon):
    """
    Apply the Harvey correction to a rectangular-kernel DM statistic.
    """
    diff = np.asarray(first) - np.asarray(second)
    size = len(diff)
    if size <= horizon:
        return np.nan, np.nan
    centered = diff - diff.mean()
    long_var = np.dot(centered, centered) / size
    for lag in range(1, horizon):
        long_var += 2 * np.dot(centered[:-lag], centered[lag:]) / size
    if long_var <= 0:
        return np.nan, np.nan
    correction = np.sqrt((size + 1 - 2 * horizon + horizon * (horizon - 1) / size) / size)
    statistic = diff.mean() / np.sqrt(long_var / size) * correction
    return statistic, 2 * student_t.sf(abs(statistic), df=size - 1)


def compare_depths(losses):
    rows = []
    for horizon, sample in losses.groupby("horizon"):
        scopes = [("Panel", sample)] + list(sample.groupby("asset"))
        for scope, frame in scopes:
            for metric in ("MSE", "QLIKE"):
                series = frame.pivot_table(index="date", columns="model", values=metric)
                for shallow, deep in (("GNNHAR1LQ", "GNNHAR2LQ"), ("GNNHAR1LQ", "GNNHAR3LQ")):
                    statistic, p_value = dm_test(series[deep], series[shallow], horizon)
                    rows.append((horizon, scope, metric, deep, shallow, statistic, p_value))
    return pd.DataFrame(
        rows, columns=["horizon", "scope", "loss", "first", "second", "DM", "p_value"]
    )


def stationary_bootstrap_means(values, draws, block, rng):
    """
    Resample time rows jointly across models with a geometric block length.
    """
    size, models = values.shape
    means = np.empty((draws, models))
    for start in range(0, draws, 128):
        stop = min(start + 128, draws)
        candidates = rng.integers(size, size=(size, stop - start))
        reset = rng.random(candidates.shape) < 1 / block
        for row in range(1, size):
            candidates[row] = np.where(reset[row], candidates[row], (candidates[row - 1] + 1) % size)
        means[start:stop] = values[candidates].mean(axis=0)
    return means


def model_confidence_set(losses, horizon, draws=1000, alpha=0.05):
    """
    Eliminate inferior models using the SQ statistic on daily panel losses.
    """
    values = losses.to_numpy()
    observed = values.mean(axis=0)
    bootstrap = stationary_bootstrap_means(values, draws, max(2, horizon), np.random.default_rng(42))
    centered = bootstrap - observed
    remaining = list(range(values.shape[1]))

    def studentize(obs, sims):
        std = np.sqrt(np.mean(sims ** 2, axis=0))
        default = np.where(obs > 0, np.inf, np.where(obs < 0, -np.inf, 0.0))
        return (
            np.divide(obs, std, out=default, where=std > 0),
            np.divide(sims, std, out=np.zeros_like(sims), where=std > 0),
        )

    while len(remaining) > 1:
        lab = np.asarray(remaining)
        pairs = np.triu_indices(len(lab), k=1)
        a, b = lab[pairs[0]], lab[pairs[1]]
        z, boot_z = studentize(observed[a] - observed[b], centered[:, a] - centered[:, b])
        p_value = (1 + (np.sum(boot_z ** 2, axis=1) >= np.sum(z ** 2)).sum()) / (draws + 1)
        if p_value >= alpha:
            break
        scores, _ = studentize(
            observed[lab] - observed[lab].mean(),
            centered[:, lab] - centered[:, lab].mean(axis=1, keepdims=True),
        )
        remaining.pop(int(np.argmax(scores)))
    return list(losses.columns[remaining])


def regime_flags(variance, horizon, source, data_dir):
    """
    Label realized stress after the fact using a horizon-matched market series.
    """
    if source == "oxford":
        reference = pd.DataFrame({".SPX": variance[".SPX"]})
    else:
        path = data_dir / "SPY_var_FH1.csv"
        if not path.exists():
            return None
        reference = read_wide(path)[["SPY"]].reindex(variance.index)
    series = pd.Series(forward_variance(reference, horizon).ravel(), index=variance.index)
    return pd.Series(np.where(series >= series.quantile(0.9), "Stress", "Calm"), index=series.index).where(series.notna())


def stratify(losses, variance, source, data_dir):
    baseline = losses.loc[losses.model == "HARM", ["date", "asset", "horizon", "forecast"]]
    compared = losses.merge(
        baseline.rename(columns={"forecast": "baseline"}), on=["date", "asset", "horizon"],
        validate="many_to_one",
    )
    compared["center"] = compared.groupby(["date", "horizon", "model"])["forecast"].transform("mean")
    compared["fvu_numerator"] = (compared.forecast - compared.baseline) ** 2
    compared["fvu_denominator"] = (compared.forecast - compared.center) ** 2
    keys = ["horizon", "model", "date"]
    daily = compared.groupby(keys)[["fvu_numerator", "fvu_denominator"]].sum()
    daily["FVU"] = daily.fvu_numerator.div(daily.fvu_denominator.where(daily.fvu_denominator > 0))
    labeled = []
    for horizon, sample in compared.groupby("horizon"):
        flags = regime_flags(variance, horizon, source, data_dir)
        if flags is not None:
            labeled.append(sample.assign(regime=sample.date.map(flags)))
    if not labeled:
        empty = pd.DataFrame(columns=["horizon", "regime", "model", "MSE", "QLIKE", "FVU"])
        return empty
    sample = pd.concat(labeled)
    keys = ["horizon", "regime", "model"]
    mean = sample.groupby(keys)[["MSE", "QLIKE"]].mean()
    relative = mean.div(mean.xs("HARM", level="model").reindex(mean.index.droplevel("model")).to_numpy())
    groups = sample[["horizon", "model", "date", "regime"]].drop_duplicates()
    fvu = daily.reset_index().merge(groups).groupby(keys)["FVU"].mean()
    return relative.join(fvu).reset_index()

In [10]:
table_1 = {
    "HARM":     [1.000, 1.000, 1.000, 1.000, 1.000, 1.000],
    "GHARM":    [0.927, 0.983, 0.904, 0.987, 0.975, 1.036],
    "GNNHAR1LM":[0.907, 0.979, 0.940, 0.943, 1.021, 0.968],
    "GNNHAR2LM":[0.967, 0.977, 1.034, 0.953, 1.134, 1.032],
    "GNNHAR3LM":[1.210, 0.982, 1.014, 0.961, 1.046, 0.958],
    "HARQ":     [0.927, 0.981, 0.939, 0.945, 1.069, 0.986],
    "GHARQ":    [0.886, 0.983, 0.842, 0.936, 1.151, 0.954],
    "GNNHAR1LQ":[0.867, 0.961, 0.855, 0.913, 1.179, 0.965],
    "GNNHAR2LQ":[0.879, 0.959, 0.873, 0.920, 1.736, 0.947],
    "GNNHAR3LQ":[0.894, 0.963, 1.185, 0.942, 1.502, 0.971],
}
columns = pd.MultiIndex.from_product([(1, 5, 22), ("MSE", "QLIKE")], names=("horizon", "loss"))
published = pd.DataFrame.from_dict(table_1, orient="index", columns=columns)
losses = add_losses(forecast)
observed = summarize_losses(losses).reindex(index=published.index, columns=published.columns)
print(f"Observed · {SOURCE} · relative to HARM")
display(observed.round(3))
print("Published · LOBSTER stocks · Table 1")
display(published)
if SOURCE == "lobster":
    print("Observed minus published")
    display((observed - published).round(3))

runner = observed.idxmin()
display(pd.DataFrame({"lowest_loss_model": runner, "ratio": observed.min()}).round(3))
heatmap = go.Figure(go.Heatmap(
    z=observed.to_numpy(), y=observed.index.tolist(),
    x=[f"{h} sessions · {metric}" for h, metric in observed.columns],
    text=observed.round(3).to_numpy(), texttemplate="%{text:.3f}",
    colorscale="RdBu_r", zmid=1, colorbar={"title": "Loss ratio"},
))
heatmap.update_layout(
    title=f"{SOURCE.upper()} · forecast loss relative to HAR-MSE",
    template="plotly_white", yaxis={"autorange": "reversed"}, height=450,
)
heatmap.show(renderer="plotly_mimetype")

Observed · clements · relative to HARM


horizon       1             5             22       
loss         MSE  QLIKE    MSE  QLIKE    MSE  QLIKE
HARM       1.000  1.000  1.000  1.000  1.000  1.000
GHARM      0.914  1.092  0.859  0.992  0.955  1.002
GNNHAR1LM  0.925  0.442  0.894  0.960  1.025  1.017
GNNHAR2LM  0.921  0.442  0.899  0.953  1.025  1.007
GNNHAR3LM  0.938  0.441  0.906  0.951  1.029  1.007
HARQ       0.923  0.436  0.880  0.921  1.208  0.938
GHARQ      0.883  0.430  0.849  0.899  1.216  0.927
GNNHAR1LQ  0.882  0.432  0.840  0.905  1.209  0.911
GNNHAR2LQ  0.882  0.432  0.843  0.902  1.195  0.907
GNNHAR3LQ  0.890  0.432  0.846  0.905  1.194  0.904

Published · LOBSTER stocks · Table 1


horizon       1             5             22       
loss         MSE  QLIKE    MSE  QLIKE    MSE  QLIKE
HARM       1.000  1.000  1.000  1.000  1.000  1.000
GHARM      0.927  0.983  0.904  0.987  0.975  1.036
GNNHAR1LM  0.907  0.979  0.940  0.943  1.021  0.968
GNNHAR2LM  0.967  0.977  1.034  0.953  1.134  1.032
GNNHAR3LM  1.210  0.982  1.014  0.961  1.046  0.958
HARQ       0.927  0.981  0.939  0.945  1.069  0.986
GHARQ      0.886  0.983  0.842  0.936  1.151  0.954
GNNHAR1LQ  0.867  0.961  0.855  0.913  1.179  0.965
GNNHAR2LQ  0.879  0.959  0.873  0.920  1.736  0.947
GNNHAR3LQ  0.894  0.963  1.185  0.942  1.502  0.971

lowest_loss_model  ratio
horizon loss                          
1       MSE           GNNHAR1LQ  0.882
        QLIKE             GHARQ  0.430
5       MSE           GNNHAR1LQ  0.840
        QLIKE             GHARQ  0.899
22      MSE               GHARM  0.955
        QLIKE         GNNHAR3LQ  0.904

**Saved 30-stock Clements–Perera run.** Relative to HAR-MSE, one-layer GNNHAR trained with QLIKE lowered pooled MSE by 11.8% at one session and 16.0% at five sessions, and raised it by 20.9% at 22 sessions. GHAR-MSE had the lowest 22-session MSE. These ratios use the public Refinitiv-based panel and the two-seed run. The published LOBSTER ratios above are a separate benchmark.

**Earlier four-index Oxford-Man benchmark.** The GNNHAR1L-QLIKE one-session MSE was 0.952 times the HAR-MSE benchmark, compared with the paper's 0.867. The smallest one-session MSE in the public run came from GNNHAR3L-QLIKE (0.951), nearly tied with the one-layer model. GHAR-QLIKE had the smallest one-session QLIKE. The public panel therefore supports an incremental short-horizon benefit in this run, with a different model ranking.

The very low public one-session QLIKE ratios depend on the denominator. For the S&P 500 on 2 April 2020, HAR-MSE forecast 0.012 percent squared against 7.577 realized. That single asset-date contributed about 5% of the entire baseline's one-session QLIKE. Around-zero forecasts deserve a separate review when interpreting this metric. At 22 sessions, the one-layer model fitted with MSE had a QLIKE ratio above 11 despite a much smaller change in MSE. Select a model against the loss and horizon that matter for the use case.

In [11]:
dm = compare_depths(losses)
display(dm.loc[dm.scope.eq("Panel")].round(4))

mcs = []
for horizon, sample in losses.groupby("horizon"):
    for metric in ("MSE", "QLIKE"):
        daily_loss = sample.pivot_table(index="date", columns="model", values=metric)
        draws = 10_000 if TRAIN_FULL_ENSEMBLE else 1_000
        survivors = model_confidence_set(daily_loss, horizon, draws=draws)
        mcs.append((horizon, metric, ", ".join(survivors)))
display(pd.DataFrame(mcs, columns=["horizon", "loss", "95% Model Confidence Set"]))

,horizon,scope,loss,first,second,DM,p_value
0,1,Panel,MSE,GNNHAR2LQ,GNNHAR1LQ,0.0164,0.9869
1,1,Panel,MSE,GNNHAR3LQ,GNNHAR1LQ,1.8377,0.0662
2,1,Panel,QLIKE,GNNHAR2LQ,GNNHAR1LQ,0.2999,0.7643
3,1,Panel,QLIKE,GNNHAR3LQ,GNNHAR1LQ,2.0388,0.0416
124,5,Panel,MSE,GNNHAR2LQ,GNNHAR1LQ,0.2692,0.7878
125,5,Panel,MSE,GNNHAR3LQ,GNNHAR1LQ,0.4032,0.6869
126,5,Panel,QLIKE,GNNHAR2LQ,GNNHAR1LQ,-1.6060,0.1084
127,5,Panel,QLIKE,GNNHAR3LQ,GNNHAR1LQ,-0.0223,0.9823
248,22,Panel,MSE,GNNHAR2LQ,GNNHAR1LQ,-2.2354,0.0255
249,22,Panel,MSE,GNNHAR3LQ,GNNHAR1LQ,-2.1637,0.0306


,horizon,loss,95% Model Confidence Set
0,1,MSE,"GHARM, GHARQ, GNNHAR1LM, GNNHAR1LQ, GNNHAR2LM,..."
1,1,QLIKE,GHARQ
2,5,MSE,"GHARM, GHARQ, GNNHAR1LM, GNNHAR1LQ, GNNHAR2LM,..."
3,5,QLIKE,"GHARQ, GNNHAR1LQ, GNNHAR2LQ, GNNHAR3LQ"
4,22,MSE,"GHARM, GHARQ, GNNHAR1LM, GNNHAR1LQ, GNNHAR2LM,..."
5,22,QLIKE,"GHARM, GHARQ, GNNHAR1LM, GNNHAR1LQ, GNNHAR2LM,..."


## 6. Stress regimes and model diagnostics

Stress uses the upper decile of horizon-matched market RV. The public run uses the S&P 500 index; the paper uses SPY. The Clements–Perera panel has no separate market series for this split. These labels are known after the forecast and are only used for analysis. A production alert would instead use a threshold estimated with information available at the forecast date.

FVU (paper Eq. 11) divides the squared distance from HAR-MSE forecasts by the cross-sectional dispersion of the candidate model's forecasts around their own mean. FVU above 1 is possible. It measures how differently a model predicts, not whether that difference benefits forecasting. MAD (Eq. 12) measures positive cosine distances between neighboring final embeddings, after averaging the ensemble embeddings. Pairs with zero norms or effectively zero distance produce no positive distance. Lower MAD can indicate over-smoothing; check it together with the depth comparison and edge count.

In [12]:
regimes = stratify(losses, variance, SOURCE, DATA_DIR)
selected = ["HARM", "GHARM", "GNNHAR1LM", "HARQ", "GHARQ", "GNNHAR1LQ", "GNNHAR2LQ", "GNNHAR3LQ"]
if len(regimes):
    display(regimes.loc[regimes.horizon.eq(1) & regimes.model.isin(selected)].round(3))
elif SOURCE == "lobster":
    print("To evaluate market regimes, supply data/lobster/SPY_var_FH1.csv.")
else:
    print("The Clements–Perera files do not include the market reference for regime stratification.")

daily_mad = forecast.drop_duplicates(["horizon", "model", "date"])
depth_models = ["GNNHAR1LQ", "GNNHAR2LQ", "GNNHAR3LQ"]
depth = daily_mad.loc[daily_mad.horizon.eq(1) & daily_mad.model.isin(depth_models)]
display(depth.groupby("model").mad.agg(["count", "median", "mean"]).round(4))

The Clements–Perera files do not include the market reference for regime stratification.


,count,median,mean
model,,,
GNNHAR1LQ,2484,0.0088,0.0129
GNNHAR2LQ,2484,0.0007,0.0021
GNNHAR3LQ,2483,0.0001,0.0011


## 7. Use cases and an observed stress episode

1. **Daily risk monitoring.** Compare an asset's HAR-only forecast with a graph model at the close. A large forecast increase when the graph branch is enabled flags a name worth investigating alongside its peers. The chart reports the graph branch's model sensitivity, with weights and own-history inputs held fixed.
2. **Weekly stress planning.** Compare 5-session forecast losses for HAR and GHAR/GNNHAR. If graph features consistently help on the portfolio's own out-of-sample data, the 5-session variance can inform a review of risk limits during a volatile week.
3. **Monthly planning.** Use the 22-session row to decide whether graph complexity earns its cost at the longer horizon. The model predicts each asset's marginal variance. Portfolio variance also needs cross-asset covariances; a GLASSO adjacency is not a covariance forecast.

The next cell shows the March 2020 stress test for the Dow in Oxford-Man mode, Apple in licensed mode, or the first numbered stock in Clements–Perera mode. Turning off peer edges is a model sensitivity experiment, not evidence that a specific neighbor caused a shock. Realized variance is plotted in percent squared; square roots give volatility in percent over the corresponding horizon.

In [13]:
asset = {"oxford": ".DJI", "lobster": "AAPL", "clements": "stock_01"}[SOURCE]
episode = forecast.loc[
    forecast.horizon.eq(1) & forecast.asset.eq(asset)
    & forecast.date.between("2020-02-18", "2020-04-30")
]
if episode.empty:
    raise ValueError("The configured run does not contain the March 2020 stress example.")

graph_case = episode.loc[episode.model.eq("GNNHAR1LQ")].set_index("date")
har_case = episode.loc[episode.model.eq("HARM")].set_index("date")
cases = pd.DataFrame({
    "Realized": graph_case.actual,
    "HAR-MSE": har_case.forecast,
    "GNNHAR1L-QLIKE": graph_case.forecast,
    "GNNHAR1L-QLIKE, peers off": graph_case.peers_off,
})
delta = graph_case.forecast - graph_case.peers_off
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.72, 0.28], vertical_spacing=0.08)
for label, color in zip(cases, ("#31363b", "#e48237", "#087f8c", "#8e92a4")):
    fig.add_trace(go.Scatter(
        x=cases.index, y=cases[label], name=label, line={"color": color, "width": 2},
    ), row=1, col=1)
fig.add_trace(go.Bar(x=delta.index, y=delta, name="Peer effect", marker_color="#087f8c"), row=2, col=1)
fig.add_hline(y=0, line_width=1, line_color="#31363b", row=2, col=1)
fig.update_yaxes(title_text="Variance (%²)", row=1, col=1)
fig.update_yaxes(title_text="Peer effect (%²)", row=2, col=1)
fig.update_xaxes(title_text="Forecast origin", row=2, col=1)
fig.update_layout(
    title=f"{asset} · one-session realized variance during the 2020 selloff",
    template="plotly_white", hovermode="x unified", height=580,
    legend={"orientation": "h", "y": 1.07},
)
fig.show(renderer="plotly_mimetype")

largest = delta.abs().idxmax()
display(cases.assign(peer_effect=delta).loc[[largest]].round(3))

,Realized,HAR-MSE,GNNHAR1L-QLIKE,"GNNHAR1L-QLIKE, peers off",peer_effect
date,,,,,
2020-03-19,39.139,24.332,30.082,23.94,6.143


## 8. Reproduction checklist and team handoff

1. Compare licensed panel coverage and per-stock RV statistics with Appendix A. Confirm both return and variance units and the calendar before fitting.
2. Run the ten-seed profile with the same forecasts and labels across models. Compare the Table 1 ratios, model ranking, and stress behavior. For numerical reconstruction, use the published and released-code choices documented above as separate specifications.
3. An independent collaborator can audit the quote-to-RV calculation on several licensed sessions and compare the validation boundary and aggregate scores. Record the observed differences and open a PR for the audit or resulting fix. Attribute any jointly written commit using the collaborator's chosen GitHub-linked address. [GitHub co-author guidance](https://docs.github.com/en/pull-requests/how-tos/commit-changes/creating-a-commit-with-multiple-authors)

Scope: the main DJIA design, a LOBSTER measurement check, public stock and index backtests, model-depth comparison, and a March 2020 forecast example. The paper's S&P 100, alternate validation, and other appendix robustness experiments are outside the saved run. The precise original inputs and configuration are required to assess exact empirical reproducibility.

In [14]:
if EXPORT_RESULTS:
    output = Path("results") / SOURCE
    output.mkdir(parents=True, exist_ok=True)
    forecast.to_csv(output / "forecasts.csv", index=False)
    audit.to_csv(output / "audit.csv", index=False)
    observed.to_csv(output / "loss_ratios.csv")
    dm.to_csv(output / "dm_tests.csv", index=False)
    regimes.to_csv(output / "regimes.csv", index=False)
    print(f"Wrote results to {output}")